# Hive adapter v2 -- DoRA training

Trains the v2 adapter for `Agent.decide()` (TRAINING_SPEC.md §11) on `fine_tune_v2/data/`:
runtime-rendered prompts, solver-built gold responses, served and trained through the same chat
helper (`serving.py`: one user turn, no system turn, thinking off, loss on the response only).

**Before you start:** GPU **T4 x2** (or P100), **Internet on**. No inputs to attach -- code, data and
the training script come from the GitHub clone, the base model from Hugging Face.

**Runs, in order (TRAINING_SPEC §11, D8).** `JOBS` holds one or two jobs; with two, each trains as an
independent process on its own T4 (same code, data, seed and memory as alone), so a session does
twice the work. The second job starts after the first has loaded the model. Check that each job's
`tok/s` stays near the single-job figure (~146 for DoRA); if it drops clearly, run one job at a time.
1. `RUN = 'smoke'` -- 30 updates on 10% of the rows, the longest row first. Read the printed `tok/s`,
   `mem` and skipped fp16 steps: they set the cost of everything after. (~30-60 min)
2. `RUN = 'sweep'` -- seed 0, 25% of the rows, LR 5e-5 / 1e-4 / 2e-4: session A runs two LRs in
   parallel, session B the third. Keep the LR with the best dev `action_match` (then lowest `dev_loss`).
3. `RUN = 'full'` -- the chosen LR: session A seeds 0 + 1 in parallel, session B seed 2.

Outputs land in `/kaggle/working/adapters/`; the last cell zips them. A session that dies keeps
`last/`: re-run with `RESUME = True` to continue from the exact update.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed: {r.stdout} {r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('synced to:', run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip())
for f in ('fine_tuning.py', 'serving.py', 'fine_tune_v2/data/train.jsonl', 'fine_tune_v2/data/dev.jsonl'):
    assert os.path.exists(f), f'push {f} first'

In [ ]:
!pip -q install -U transformers peft accelerate

## Config

In [ ]:
RUN = 'sweep'          # 'smoke' | 'sweep' | 'full'
# One job per GPU, at most two. Each job: adapter ('dora' | 'lora'), lr, seed.
JOBS = [
    dict(adapter='dora', lr=5e-5, seed=0),
    dict(adapter='dora', lr=1e-4, seed=0),
]
RESUME = False         # True: every job continues from its own last/ checkpoint
BASE = 'Qwen/Qwen3-4B'
REVISION = None        # S5: set to the pinned commit hash once chosen (same in serving)
MAX_LENGTH = 3072      # data's longest row is in fine_tune_v2/data/MANIFEST.json (max_total_tokens)

PLANS = {
    # smoke: >= 30 real updates, the longest row first (peak memory), small evals
    'smoke': dict(subset=0.10, accum=4, max_updates=30, checkpoint_every=15, dev_gen=10, longest=True),
    'sweep': dict(subset=0.25, accum=16, max_updates=None, checkpoint_every=20, dev_gen=30, longest=False),
    'full':  dict(subset=1.0, accum=16, max_updates=None, checkpoint_every=25, dev_gen=60, longest=False),
}
plan = PLANS[RUN]
assert 1 <= len(JOBS) <= 2, 'one job per GPU, at most two'
for j in JOBS:
    j['out'] = f"/kaggle/working/adapters/v2-{j['adapter']}-{RUN}-lr{j['lr']:g}-s{j['seed']}"
    print(j['out'])
assert len({j['out'] for j in JOBS}) == len(JOBS), 'two jobs would write to the same folder'
print(plan)

In [ ]:
import json, torch
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('native bf16:', torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
      '(T4 / P100 train in fp16 with loss scaling)')
m = json.load(open('fine_tune_v2/data/MANIFEST.json'))
print('data:', m['counts'], '| longest row:', m['max_total_tokens'], 'tokens | built at', m['generator_commit'])
print('classes:', m['class_counts']['train'])
assert m['max_total_tokens'] < MAX_LENGTH, 'raise MAX_LENGTH above the longest row'
assert torch.cuda.device_count() >= len(JOBS), f'{len(JOBS)} jobs need {len(JOBS)} GPUs'
# The locked test is sealed (TRAINING_SPEC §9): this notebook never reads fine_tune_v2/locked/.

## Train

In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, 'fine_tune_v2'))
import parallel

def job_cmd(j):
    cmd = [sys.executable, 'fine_tuning.py', '--adapter', j['adapter'], '--data', 'fine_tune_v2/data',
           '--out', j['out'], '--base', BASE, '--seed', str(j['seed']), '--lr', str(j['lr']),
           '--max-length', str(MAX_LENGTH), '--subset', str(plan['subset']),
           '--checkpoint-every', str(plan['checkpoint_every']), '--dev-gen', str(plan['dev_gen']),
           '--accum', str(plan['accum'])]
    if REVISION:
        cmd += ['--revision', REVISION]
    if plan['max_updates']:
        cmd += ['--max-updates', str(plan['max_updates'])]
    if plan['longest']:
        cmd += ['--include-longest']
    if RESUME:
        cmd += ['--resume']
    return cmd

jobs = [dict(name=f"gpu{g} {j['adapter']} lr{j['lr']:g} s{j['seed']}", gpu=g, cmd=job_cmd(j))
        for g, j in enumerate(JOBS)]
codes = parallel.run_jobs(jobs)
assert all(c == 0 for c in codes.values()), f'a job failed -- read its [gpuN] lines above: {codes}'

## Read the run

`tok/s` and peak `mem` decide the cost of the sweep and seed runs (TRAINING_SPEC §11, W7).
`action_match` / `payload_valid` / `stopped` are cheap dev metrics for checkpoint choice; the §10
gates run in the evaluation harness, not here.

In [ ]:
for j in JOBS:
    rec = json.load(open(os.path.join(j['out'], 'TRAINING_RECORD.json')))
    print('====', os.path.basename(j['out']))
    print(f"updates {rec['updates']} | skipped fp16 steps {rec['skipped_fp16_steps']} | {rec['dtype']} | "
          f"tok/s {rec['tokens_per_second']:.0f} (training time only) | peak {rec['peak_memory_gib']:.1f} GiB")
    print('chosen checkpoint:', rec['chosen_checkpoint'])
    for h in rec['history']:
        print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in h.items()})

## Spot-check three dev prompts with the first job's best adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import fine_tuning
fine_tuning.disable_peft_torchao_probe()  # Kaggle's torchao 0.10 vs current peft
from peft import PeftModel
import serving
tok = AutoTokenizer.from_pretrained(BASE)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, device_map={'': 0})
OUT = JOBS[0]['out']  # spot-check the first job's best adapter
print('spot-checking', OUT)
model = PeftModel.from_pretrained(base, os.path.join(OUT, 'best')).eval()
dev = [json.loads(l) for l in open('fine_tune_v2/data/dev.jsonl')][:3]
stop = [tok.convert_tokens_to_ids('<|im_end|>'), tok.convert_tokens_to_ids('<|endoftext|>')]
for r in dev:
    ids = torch.tensor([serving.render(tok, r['prompt'])], device=model.get_input_embeddings().weight.device)
    with torch.autocast('cuda', dtype=dtype):
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=400,
                             do_sample=False, eos_token_id=stop, pad_token_id=tok.pad_token_id)
    print('====', r.get('class'), r.get('family'))
    print('GOLD:', r['response'][:400])
    print('GEN :', tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)[:400])

## Download

In [ ]:
import shutil
# last/ holds optimiser state for resume (large); ship only best/ and the records.
pack = '/kaggle/working/adapters_pack'
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree('/kaggle/working/adapters', pack, ignore=shutil.ignore_patterns('last', 'last.tmp', '*.tmp'))
shutil.make_archive('/kaggle/working/adapters', 'zip', pack)
print('download /kaggle/working/adapters.zip')
